# Bridging Genetics and Precision Medicine in Parkinson’s Disease through GP2 

```GP2 ❤️ Open Science 😍```  
**Project:** Bridging Genetics and Precision Medicine in Parkinson’s Disease through GP2  
**Last Updated:** 20-MAR-2026

## Set-up

In [ ]:
# install packages
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from re import search
import os
import glob
import sys

In [ ]:
# Path to GP2 Tier 2 Release 11 clinical data directory

clin_data_path = '/path/to/gp2_release11/clinical_data/'

# Path to data analysis directory
datafolder = '/path/to/LRRK2_GBA_PM/'

## Precision medicine table

### Total numbers

Inclusion criteria:


*   PD cases with LRRK2 and/or GBA1 mutations
*   In R11
*   With either WGS, NBA, or CES





Exclusion criteria:

*   Brain banks
*   People with age_at_death
*   Non-PD cases




Information:

*   Disease duration was calculated using the age at enrollment and AAO, or age at diagnosis, if AAO was unavailable.
*   Clinical variables of direct relevance to clinical trial eligibility, including disease duration, Hoehn & Yahr stage (H&Y), and Montreal Cognitive Assessment (MoCA) scores were extracted from the most recent GP2 data releases, where available.



In [ ]:
df = pd.read_excel(f"{datafolder}GP2_R11_GBA1_LRRK2_updated.xlsx")
df["Group"].unique()

array(['GBA1', 'LRRK2', 'GBA1 + dual', 'LRRK2 + dual', 'EX', 'LRRK2 risk',
       'LRRK2 risk + dual', 'two_GBA1', 'two_LRRK2', 'two_LRRK2_risk',
       'two_LRRK2_risk + dual', 'two GBA1 + dual'], dtype=object)

In [ ]:
# To take into account the exclusion criteria

df = pd.read_excel(f"{datafolder}GP2_R11_GBA1_LRRK2_updated.xlsx")
df = df[
    ~df["study_type"].isin(["Brain Bank"]) &
    df["baseline_GP2_phenotype"].str.lower().str.contains("pd", na=False)
]

# Load masterkey for age at death filtering
vital = pd.read_csv(f"{clin_data_path}master_key_release11_final_vwb.csv")
dead_ids = vital.loc[vital["age_at_death"].notna(), "GP2ID"]
df = df[~df["GP2ID"].isin(dead_ids)]
df = df.drop_duplicates(subset="GP2ID")
df["Ancestry"] = df["Ancestry"].fillna("Other")

print(f"Final N (unique GP2ID): {df['GP2ID'].nunique()}")
df.to_csv(f"{datafolder}lara_pm_ids_filtered.csv")


Final N (unique GP2ID): 9158


In [ ]:
assert df['GP2ID'].is_unique, "Duplicate GP2IDs still present"


In [ ]:
# Define mutation groups and total count for each per ancestry

# Define carrier flags
df["GBA_carrier"] = df["Group"].str.contains("GBA1", na=False) & ~df["Group"].str.contains("dual", na=False)
df["LRRK2_carrier"] = df["Group"].str.contains("LRRK2", na=False) & ~df["Group"].str.contains("dual", na=False)
df["GBA_LRRK2_carrier"] = df["Group"].str.contains("dual", na=False)

# Group by ancestry and summarise
summary = (
    df.groupby("Ancestry")
      .agg(
          n_participants=("Group", "count"),
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reset_index()
)

print(summary)

# Calculate total unique IDs across GBA, LRRK2, and GBA_LRRK2 groups
total_ids = summary['n_GBA'].sum() + summary['n_LRRK2'].sum() + summary['n_GBA_LRRK2'].sum()
print(f"\nTotal IDs in GBA, LRRK2, and GBA_LRRK2 groups: {total_ids}")

   Ancestry  n_participants  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC             202    197        0            1
1       AFR            1504   1485        3            2
2        AJ             989    469      465           54
3       AMR             235    165       60            2
4       CAH             187    146       32            3
5       CAS             104     76       24            1
6       EAS             822    192      598           20
7       EUR            4181   3420      633           43
8       FIN              27     27        0            0
9       MDE             158     67       79            8
10    Other             704    503      189           12
11      SAS              45     42        1            0

Total IDs in GBA, LRRK2, and GBA_LRRK2 groups: 9019


In [ ]:
# Subset to GBA + LRRK2 (dual) carriers
gba_lrrk2_df = df.loc[df["GBA_LRRK2_carrier"]].copy()
gba_lrrk2_df.to_excel(f"{datafolder}dual_carriers_r11.xlsx", index=False)
print(f"Saved {gba_lrrk2_df['GP2ID'].nunique()} unique GBA+LRRK2 carrier IDs")

# check just lrrk2
lrrk2_df = df.loc[df["LRRK2_carrier"]].copy()
lrrk2_df.to_excel(f"{datafolder}lrrk2_carriers_r11.xlsx", index=False)
print(f"Saved {lrrk2_df['GP2ID'].nunique()} unique LRRK2 carrier IDs")

# check just gba1
gba_df = df.loc[df["GBA_carrier"]].copy()
gba_df.to_excel(f"{datafolder}GBA1_carriers_r11.xlsx", index=False)
print(f"Saved {gba_df['GP2ID'].nunique()} unique GBA carrier")

Saved 146 unique GBA+LRRK2 carrier IDs
Saved 2084 unique LRRK2 carrier IDs
Saved 6789 unique GBA carrier


## Clinical data availability

In [ ]:
# N carriers with clinical data (based on R11)

df_genetic_info = pd.read_csv(f"{datafolder}lara_pm_ids_filtered.csv")

clin_raw = pd.read_csv(f"{clin_data_path}r11_extended_clinical_data.csv", low_memory=False)
clin = clin_raw.copy()

clinical_data_cols = clin.columns.difference(
    ["GP2ID", "visit_month", "date_visit_unix", "date_enrollment"]
)

clin["has_clinical_data"] = clin[clinical_data_cols].notna().any(axis=1)

clin_id_level = (
    clin.groupby("GP2ID")["has_clinical_data"]
        .any()
        .reset_index(name="has_clinical_data")
)

df["GP2ID"] = df["GP2ID"].astype(str)
clin_id_level["GP2ID"] = clin_id_level["GP2ID"].astype(str)

df = df.merge(
    clin_id_level,
    on="GP2ID",
    how="left"
)

df["has_clinical_data"] = df["has_clinical_data"].fillna(False)

df["GBA_carrier"] = (
    df["Group"].str.contains("GBA1", na=False) &
    ~df["Group"].str.contains("dual", na=False)
)

df["LRRK2_carrier"] = (
    df["Group"].str.contains("LRRK2", na=False) &
    ~df["Group"].str.contains("dual", na=False)
)

df["GBA_LRRK2_carrier"] = df["Group"].str.contains("dual", na=False)

summary = (
    df[df["has_clinical_data"]]
    .drop_duplicates(subset="GP2ID")
    .groupby("Ancestry")
    .agg(
        n_GBA=("GBA_carrier", "sum"),
        n_LRRK2=("LRRK2_carrier", "sum"),
        n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
    )
    .reset_index()
)

print(summary)

#checks
df[df["has_clinical_data"]]["GP2ID"].nunique()

df.shape
df["GP2ID"].value_counts().head()


n_ids_with_clin_data_genetic = (
    df.loc[
        df["has_clinical_data"] &
        (df["GBA_carrier"] | df["LRRK2_carrier"] | df["GBA_LRRK2_carrier"]),
        "GP2ID"
    ]
    .nunique()
)

print(f"Total number of IDs with clinical data: {n_ids_with_clin_data_genetic}")
print("Total genetic IDs:", df["GP2ID"].nunique())
print("Genetic IDs with clinical data:", n_ids_with_clin_data_genetic)

print(
    "Should equal sum across ancestry rows:",
    summary[["n_GBA", "n_LRRK2", "n_GBA_LRRK2"]].sum().sum()
)


   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC    106        0            0
1       AFR    156        0            1
2        AJ    284      228           25
3       AMR     39       16            1
4       CAH     64        9            1
5       CAS      7        3            0
6       EAS     85      142            8
7       EUR   1233      280           19
8       FIN      9        0            0
9       MDE     13       17            1
10    Other     48       16            3
11      SAS     18        0            0
Total number of IDs with clinical data: 2832
Total genetic IDs: 9158
Genetic IDs with clinical data: 2832
Should equal sum across ancestry rows: 2832


/tmp/ipython-input-1814032036.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_clinical_data"] = df["has_clinical_data"].fillna(False)


In [ ]:
print("\nAll columns in clinical dataset:")
for c in clin.columns:
    print(c)


All columns in clinical dataset:
Unnamed: 0
study
GP2ID
visit_month
date_visit_unix
mmse_total_score
date_enrollment
weight_kg
height_cm
BMI
heart_rate
systolic_blood_pressure
diastolic_blood_pressure
hoehn_and_yahr_stage
mds_updrs_part_i_primary_info_source
code_upd2101_cognitive_impairment
code_upd2102_hallucinations_and_psychosis
code_upd2103_depressed_mood
code_upd2104_anxious_mood
code_upd2105_apathy
code_upd2106_dopamine_dysregulation_syndrome_features
code_upd2107_pat_quest_sleep_problems
code_upd2108_pat_quest_daytime_sleepiness
code_upd2109_pat_quest_pain_and_other_sensations
code_upd2110_pat_quest_urinary_problems
code_upd2111_pat_quest_constipation_problems
code_upd2112_pat_quest_lightheadedness_on_standing
code_upd2113_pat_quest_fatigue
mds_updrs_part_i_summary_score
code_upd2201_speech
code_upd2202_saliva_and_drooling
code_upd2203_chewing_and_swallowing
code_upd2204_eating_tasks
code_upd2205_dressing
code_upd2206_hygiene
code_upd2207_handwriting
code_upd2208_doing_hobbies

## Numbers for specific data points

### Hoehn and Yahr (HY)

In [ ]:
# Number of IDs that have any data for HY

clin = pd.read_csv(
    f"{clin_data_path}r11_extended_clinical_data.csv",
    low_memory=False
)

clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)

hy_flag = (
    clin.groupby("GP2ID", as_index=False)
        .agg(has_hoehn_yahr=("hoehn_and_yahr_stage", lambda x: x.notna().any()))
)

df = df.drop(columns=["has_hoehn_yahr"], errors="ignore").merge(
    hy_flag, on="GP2ID", how="left"
)
df["has_hoehn_yahr"] = df["has_hoehn_yahr"].fillna(False)

print(f"Total IDs with Hoehn–Yahr data: {df['has_hoehn_yahr'].sum()}")

all_ancestries = (
    df.drop_duplicates("GP2ID")["Ancestry"]
      .sort_values()
      .unique()
)

hy_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_hoehn_yahr"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reindex(all_ancestries, fill_value=0) # to get all ancestries, even those without values
      .reset_index()
)

print("\nIDs with Hoehn & Yahr data by ancestry and genetic group:")
print(hy_by_ancestry_genetic)



Total IDs with Hoehn–Yahr data: 1393

IDs with Hoehn & Yahr data by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC     52        0            0
1       AFR     58        0            0
2        AJ    108      121            7
3       AMR     21       13            1
4       CAH     48        5            1
5       CAS      2        1            0
6       EAS     28       97            3
7       EUR    659      117            8
8       FIN      2        0            0
9       MDE      2        6            1
10    Other      0        0            0
11      SAS     12        0            0


/tmp/ipython-input-4221581466.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_hoehn_yahr"] = df["has_hoehn_yahr"].fillna(False)


In [ ]:
df.loc[
    (df["Ancestry"] == "Other"),
    "has_hoehn_yahr"
].value_counts(dropna=False)


,count
has_hoehn_yahr,
False,704


In [ ]:
# Number of IDs that have a HY score of 2 or below - checking the most recent score

clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)
clin_hy = clin.loc[clin["hoehn_and_yahr_stage"].notna()].copy()

# by chronological order then taking the last one
clin_hy = clin_hy.sort_values(["GP2ID", "visit_month"])
latest_hy = (
    clin_hy.groupby("GP2ID", as_index=False)
           .last()[["GP2ID", "hoehn_and_yahr_stage"]]
)

latest_hy["has_hy_le2"] = latest_hy["hoehn_and_yahr_stage"] <= 2
df = df.drop(columns=["has_hy_le2"], errors="ignore").merge(
    latest_hy[["GP2ID", "has_hy_le2"]],
    on="GP2ID",
    how="left"
)

df["has_hy_le2"] = df["has_hy_le2"].fillna(False)

n_ids_hy_le2 = df.loc[df["has_hy_le2"], "GP2ID"].nunique()
print(f"Total unique IDs with most recent Hoehn–Yahr ≤ 2: {n_ids_hy_le2}")

all_ancestries = (
    df.drop_duplicates("GP2ID")["Ancestry"]
      .sort_values()
      .unique()
)

hy_le2_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_hy_le2"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reindex(all_ancestries, fill_value=0) # to get all ancestries, even those without values
      .reset_index()
)

print("\nMost recent Hoehn & Yahr ≤ 2 by ancestry and genetic group:")
print(hy_le2_by_ancestry_genetic)



Total unique IDs with most recent Hoehn–Yahr ≤ 2: 866

Most recent Hoehn & Yahr ≤ 2 by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC     34        0            0
1       AFR     30        0            0
2        AJ     64       89            4
3       AMR     10        5            0
4       CAH     37        3            1
5       CAS      2        1            0
6       EAS     17       58            2
7       EUR    403       74            6
8       FIN      0        0            0
9       MDE      2        6            0
10    Other      0        0            0
11      SAS      7        0            0


/tmp/ipython-input-2783630670.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .last()[["GP2ID", "hoehn_and_yahr_stage"]]
/tmp/ipython-input-2783630670.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_hy_le2"] = df["has_hy_le2"].fillna(False)


In [ ]:
# Number of IDs that have a HY score of 3 or below

clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)
clin_hy = clin.loc[clin["hoehn_and_yahr_stage"].notna()].copy()

# by chronological order then taking the last one
clin_hy = clin_hy.sort_values(["GP2ID", "visit_month"])
latest_hy = (
    clin_hy.groupby("GP2ID", as_index=False)
           .last()[["GP2ID", "hoehn_and_yahr_stage"]]
)

latest_hy["has_hy_le3"] = latest_hy["hoehn_and_yahr_stage"] <= 3
df = df.drop(columns=["has_hy_le3"], errors="ignore").merge(
    latest_hy[["GP2ID", "has_hy_le3"]],
    on="GP2ID",
    how="left"
)

df["has_hy_le3"] = df["has_hy_le3"].fillna(False)

n_ids_hy_le3 = df.loc[df["has_hy_le3"], "GP2ID"].nunique()
print(f"Total unique IDs with most recent Hoehn–Yahr ≤ 3: {n_ids_hy_le3}")

all_ancestries = (
    df.drop_duplicates("GP2ID")["Ancestry"]
      .sort_values()
      .unique()
)

hy_le3_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_hy_le3"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reindex(all_ancestries, fill_value=0)  # to get all ancestries, even those without values
      .reset_index()
)

print("\nMost recent Hoehn & Yahr ≤ 3 by ancestry and genetic group:")
print(hy_le3_by_ancestry_genetic)


Total unique IDs with most recent Hoehn–Yahr ≤ 3: 1263

Most recent Hoehn & Yahr ≤ 3 by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC     43        0            0
1       AFR     51        0            0
2        AJ     93      110            7
3       AMR     19       10            1
4       CAH     47        5            1
5       CAS      2        1            0
6       EAS     25       81            3
7       EUR    608      110            6
8       FIN      2        0            0
9       MDE      2        6            1
10    Other      0        0            0
11      SAS     12        0            0


/tmp/ipython-input-2755436318.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .last()[["GP2ID", "hoehn_and_yahr_stage"]]
/tmp/ipython-input-2755436318.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_hy_le3"] = df["has_hy_le3"].fillna(False)


### MoCA

In [ ]:
# Number of IDs that have any data for moca

clin = pd.read_csv(
    f"{clin_data_path}r11_extended_clinical_data.csv",
    low_memory=False
)

clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)

# ID-level Hoehn–Yahr flag
moca_flag = (
    clin.groupby("GP2ID", as_index=False)
        .apply(lambda g: pd.Series({
            "has_moca": g[["moca_total_score", "moca_total_score_adjusted"]].notna().any().any()
        }))
)


df = df.drop(columns=["has_moca"], errors="ignore").merge(
    moca_flag, on="GP2ID", how="left"
)
df["has_moca"] = df["has_moca"].fillna(False)

print(f"Total IDs with MoCA data: {df['has_moca'].sum()}")

# Per ancestry and mutation breakdown
moca_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_moca"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reset_index()
)

print("\nIDs with MoCA data by ancestry and genetic group:")
print(moca_by_ancestry_genetic)

Total IDs with MoCA data: 1166

IDs with MoCA data by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC     32        0            0
1       AFR     24        0            0
2        AJ     95      115            9
3       AMR     22       10            1
4       CAH     35        0            1
5       CAS      1        2            0
6       EAS     31       26            0
7       EUR    472      173            5
8       FIN      4        0            0
9       MDE      3        8            1
10    Other     47       16            3
11      SAS     14        0            0


/tmp/ipython-input-2315172586.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/tmp/ipython-input-2315172586.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_moca"] = df["has_moca"].fillna(False)


In [ ]:
# Number of IDs that have a moca score of 20 or above

clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)

# Filter to rows where either MoCA score exists
clin_moca = clin.loc[
    clin["moca_total_score"].notna() | clin["moca_total_score_adjusted"].notna()
].copy()

# Sort chronologically by visit_month, then take the latest score per ID
clin_moca = clin_moca.sort_values(["GP2ID", "visit_month"])
latest_moca = (
    clin_moca.groupby("GP2ID", as_index=False)
              .last()[["GP2ID", "moca_total_score", "moca_total_score_adjusted"]]
)

# ID-level flag: most recent MoCA ≥ 20 (either score)
latest_moca["has_moca_ge20"] = (
    (latest_moca["moca_total_score"] >= 20) |
    (latest_moca["moca_total_score_adjusted"] >= 20)
)

df = df.drop(columns=["has_moca_ge20"], errors="ignore").merge(
    latest_moca[["GP2ID", "has_moca_ge20"]],
    on="GP2ID",
    how="left"
)

df["has_moca_ge20"] = df["has_moca_ge20"].fillna(False)

# Total unique IDs with MoCA ≥ 20
n_ids_moca_ge20 = df.loc[df["has_moca_ge20"], "GP2ID"].nunique()
print(f"Total unique IDs with most recent MoCA ≥ 20: {n_ids_moca_ge20}")

# All ancestries (to preserve the same order)
all_ancestries = (
    df.drop_duplicates("GP2ID")["Ancestry"]
      .sort_values()
      .unique()
)

# Per ancestry and genetic breakdown
moca_ge20_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_moca_ge20"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reindex(all_ancestries, fill_value=0)  # ensure all ancestries appear
      .reset_index()
)

print("\nMost recent MoCA ≥ 20 by ancestry and genetic group:")
print(moca_ge20_by_ancestry_genetic)


Total unique IDs with most recent MoCA ≥ 20: 1030

Most recent MoCA ≥ 20 by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC     23        0            0
1       AFR     19        0            0
2        AJ     83      108            9
3       AMR     17        9            1
4       CAH     28        0            1
5       CAS      1        2            0
6       EAS     26       25            0
7       EUR    427      147            4
8       FIN      4        0            0
9       MDE      3        4            1
10    Other     42       16            3
11      SAS     13        0            0


/tmp/ipython-input-1734505098.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .last()[["GP2ID", "moca_total_score", "moca_total_score_adjusted"]]
/tmp/ipython-input-1734505098.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_moca_ge20"] = df["has_moca_ge20"].fillna(False)


In [ ]:
# Number of IDs that have a MoCA score of 26 or above
clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)

# Filter to rows where either MoCA score exists
clin_moca = clin.loc[
    clin["moca_total_score"].notna() | clin["moca_total_score_adjusted"].notna()
].copy()

# Sort chronologically by visit_month, then take the latest score per ID
clin_moca = clin_moca.sort_values(["GP2ID", "visit_month"])
latest_moca = (
    clin_moca.groupby("GP2ID", as_index=False)
              .last()[["GP2ID", "moca_total_score", "moca_total_score_adjusted"]]
)

# ID-level flag: most recent MoCA ≥ 26
latest_moca["has_moca_ge26"] = (
    (latest_moca["moca_total_score"] >= 26) |
    (latest_moca["moca_total_score_adjusted"] >= 26)
)

df = df.drop(columns=["has_moca_ge26"], errors="ignore").merge(
    latest_moca[["GP2ID", "has_moca_ge26"]],
    on="GP2ID",
    how="left"
)

df["has_moca_ge26"] = df["has_moca_ge26"].fillna(False)

# Total unique IDs with MoCA ≥ 26
n_ids_moca_ge26 = df.loc[df["has_moca_ge26"], "GP2ID"].nunique()
print(f"Total unique IDs with most recent MoCA ≥ 26: {n_ids_moca_ge26}")

# All ancestries (to preserve the same order)
all_ancestries = (
    df.drop_duplicates("GP2ID")["Ancestry"]
      .sort_values()
      .unique()
)

# Per ancestry and genetic breakdown
moca_ge26_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_moca_ge26"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reindex(all_ancestries, fill_value=0)  # ensure all ancestries appear
      .reset_index()
)

print("\nMost recent MoCA ≥ 26 by ancestry and genetic group:")
print(moca_ge26_by_ancestry_genetic)


Total unique IDs with most recent MoCA ≥ 26: 679

Most recent MoCA ≥ 26 by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC      9        0            0
1       AFR      8        0            0
2        AJ     61       75            8
3       AMR     12        5            0
4       CAH     12        0            0
5       CAS      1        1            0
6       EAS     13       15            0
7       EUR    307       84            3
8       FIN      4        0            0
9       MDE      2        3            1
10    Other     24       15            3
11      SAS      5        0            0


/tmp/ipython-input-2951010609.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .last()[["GP2ID", "moca_total_score", "moca_total_score_adjusted"]]
/tmp/ipython-input-2951010609.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_moca_ge26"] = df["has_moca_ge26"].fillna(False)


### Disease duration

We defined disease duration based on age at enrollment and age at onset (AAO), or age at diagnosis (AAD), if AAO was unavailable.

In [ ]:
# Calculate disease duration for all carriers - based on age at enrollment and AAO, or AAD if AAO not available
clin_age = pd.read_csv(f"{clin_data_path}master_key_release11_final_vwb.csv")

clin_age["GP2ID"] = clin_age["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)

clin_age['age_at_sample_collection'] = pd.to_numeric(clin_age['age_at_sample_collection'], errors='coerce')
clin_age['age_of_onset'] = pd.to_numeric(clin_age['age_of_onset'], errors='coerce')
clin_age['age_at_diagnosis'] = pd.to_numeric(clin_age['age_at_diagnosis'], errors='coerce')

clin_age['disease_duration_calculated'] = np.where(
    clin_age['age_of_onset'].notna(),
    clin_age['age_at_sample_collection'] - clin_age['age_of_onset'],
    clin_age['age_at_sample_collection'] - clin_age['age_at_diagnosis']
)

df = df.merge(
    clin_age[['GP2ID', 'disease_duration_calculated']],
    on='GP2ID',
    how='left'
)

In [ ]:
# Number of IDs that have any data for disease duration

clin = pd.read_csv(
    f"{clin_data_path}r11_extended_clinical_data.csv",
    low_memory=False
)

clin["GP2ID"] = clin["GP2ID"].astype(str)
df["GP2ID"] = df["GP2ID"].astype(str)

# ID-level disease duration flag
duration_flag = (
    df.groupby("GP2ID", as_index=False)
      .agg(has_disease_duration=("disease_duration_calculated", lambda x: x.notna().any()))
)

df = df.drop(columns=["has_disease_duration"], errors="ignore").merge(
    duration_flag, on="GP2ID", how="left"
)

df["has_disease_duration"] = df["has_disease_duration"].fillna(False)

print(f"Total IDs with disease duration data: {df['has_disease_duration'].sum()}")

# Per ancestry and mutation breakdown
duration_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_disease_duration"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reset_index()
)


print("\nIDs with disease duration data by ancestry and genetic group:")
print(duration_by_ancestry_genetic)


Total IDs with disease duration data: 7379

IDs with disease duration data by ancestry and genetic group:
   Ancestry  n_GBA  n_LRRK2  n_GBA_LRRK2
0       AAC    145        0            1
1       AFR   1362        3            1
2        AJ    429      379           40
3       AMR     84       26            2
4       CAH    116       24            2
5       CAS     30       11            1
6       EAS    127      482           15
7       EUR   2705      547           36
8       FIN     16        0            0
9       MDE     39       72            8
10    Other    379      145            8
11      SAS     39        1            0


In [ ]:
# Number of IDs that have a disease duration <= 3 years

duration_le3_flag = (
    df.groupby("GP2ID", as_index=False)
      .apply(lambda g: pd.Series({
          "has_disease_duration_le3": (g['disease_duration_calculated'] <= 3).any()
      }))
)

df = df.drop(columns=["has_disease_duration_le3"], errors="ignore").merge(
    duration_le3_flag,
    on="GP2ID",
    how="left"
)
df["has_disease_duration_le3"] = df["has_disease_duration_le3"].fillna(False)

n_ids_duration_le3 = df.loc[df["has_disease_duration_le3"], "GP2ID"].nunique()
print(f"Total unique IDs with disease duration ≤ 3 years: {n_ids_duration_le3}")

duration_le3_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_disease_duration_le3"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reset_index()
)

print("\nDisease duration ≤ 3 years by ancestry and genetic group:")
print(duration_le3_by_ancestry_genetic)

In [ ]:
# Number of IDs that have a disease duration <= 5 years

duration_le5_flag = (
    df.groupby("GP2ID", as_index=False)
      .apply(lambda g: pd.Series({
          "has_disease_duration_le5": (g['disease_duration_calculated'] <= 5).any()
      }))
)

df = df.drop(columns=["has_disease_duration_le5"], errors="ignore").merge(
    duration_le5_flag,
    on="GP2ID",
    how="left"
)
df["has_disease_duration_le5"] = df["has_disease_duration_le5"].fillna(False)

n_ids_duration_le5 = df.loc[df["has_disease_duration_le5"], "GP2ID"].nunique()
print(f"Total unique IDs with disease duration ≤ 5 years: {n_ids_duration_le5}")

duration_le5_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_disease_duration_le5"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reset_index()
)

print("\nDisease duration ≤ 5 years by ancestry and genetic group:")
print(duration_le5_by_ancestry_genetic)


In [ ]:
# Number of IDs that have a disease duration <= 7 years

duration_le7_flag = (
    df.groupby("GP2ID", as_index=False)
      .apply(lambda g: pd.Series({
          "has_disease_duration_le7": (g['disease_duration_calculated'] <= 7).any()
      }))
)

df = df.drop(columns=["has_disease_duration_le7"], errors="ignore").merge(
    duration_le7_flag,
    on="GP2ID",
    how="left"
)
df["has_disease_duration_le7"] = df["has_disease_duration_le7"].fillna(False)

n_ids_duration_le7 = df.loc[df["has_disease_duration_le7"], "GP2ID"].nunique()
print(f"Total unique IDs with disease duration ≤ 7 years: {n_ids_duration_le7}")

duration_le7_by_ancestry_genetic = (
    df.drop_duplicates("GP2ID")
      .loc[lambda x: x["has_disease_duration_le7"]]
      .groupby("Ancestry")
      .agg(
          n_GBA=("GBA_carrier", "sum"),
          n_LRRK2=("LRRK2_carrier", "sum"),
          n_GBA_LRRK2=("GBA_LRRK2_carrier", "sum")
      )
      .reset_index()
)

print("\nDisease duration ≤ 7 years by ancestry and genetic group:")
print(duration_le7_by_ancestry_genetic)